# Group fairness diagnostics with uncertainty

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/fairness.ipynb)

Companion notebook for the ReliableML paper `docs/latex/fairness.tex`. It uses the `sensitive_features` argument of `ReliablePipeline.evaluate` to report per-group selection, error and precision rates, max-minus-min gaps and row-bootstrap intervals.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Quantities

For a positive class and groups $g\in\mathcal G$ with true label $Y$, prediction $\hat Y$ and sensitive attribute $A$ (never used as a model input here; it is only used to *audit* the predictions):
$$\mathrm{SR}_g=P(\hat Y=1\mid A=g),\quad \mathrm{TPR}_g=P(\hat Y=1\mid Y=1,A=g),\quad \mathrm{FPR}_g=P(\hat Y=1\mid Y=0,A=g),\quad \mathrm{PPV}_g=P(Y=1\mid \hat Y=1,A=g).$$
Gaps are max-minus-min across groups, $\Delta(m)=\max_g m_g-\min_g m_g$:
$$\text{statistical parity: }\Delta(\mathrm{SR}),\qquad \text{equalized odds: }\max\{\Delta(\mathrm{TPR}),\Delta(\mathrm{FPR})\},\qquad \text{predictive parity: }\Delta(\mathrm{PPV}).$$
Zero means equal *observed* rates. Each gap gets a within-group row-bootstrap percentile interval; groups with an empty denominator (for example no actual negatives) have undefined rates and are reported as such. Multiclass problems are evaluated one-vs-rest per class.

**Criteria conflict.** If base rates $P(Y=1\mid A=g)$ differ, an imperfect classifier cannot in general satisfy statistical parity, equalized odds and predictive parity at once; which gap matters is a policy choice, not a statistical one. Small gaps in a small group can be pure noise - the intervals and the denominators are part of the result.

## 2. Synthetic data with different base rates

The group `A` is a purely illustrative synthetic attribute. The model never sees it, but the feature distribution and the base rate differ by group, so error rates can differ anyway.

In [ ]:
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from ml_reliability import ReliablePipeline

warnings.filterwarnings("ignore", category=UserWarning)
rng = np.random.default_rng(3)
n = 4000
A = rng.choice(["g1", "g2"], size=n, p=[0.6, 0.4])
x0 = rng.normal(size=n)
x1 = rng.normal(size=n) + np.where(A == "g2", 0.8, 0.0)  # proxy feature shifted by group
logit = -0.2 + 1.0 * x0 + 0.5 * x1 - np.where(A == "g2", 0.8, 0.0)  # different base rate
y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
X = pd.DataFrame({"x0": x0, "x1": x1, "x2": rng.normal(size=n)})
print(pd.Series(y).groupby(A).mean().round(3).rename("base rate P(Y=1|A)"))

Xtr, Xte, ytr, yte, Atr, Ate = train_test_split(X, y, A, test_size=0.4, random_state=0, stratify=y)
pipe = ReliablePipeline(LogisticRegression(max_iter=1000), plot_dir="reliability_plots",
                        bias_variance_folds=3, bias_variance_repeats=1, bootstrap_samples=200, random_seed=0).fit(Xtr, ytr)
ev = pipe.evaluate(Xte, yte, sensitive_features={"A": Ate}, positive_label=1).to_dict()
fair = ev["fairness"]["A"]["classes"]["1"]
groups = pd.DataFrame(fair["per_group"])
print(groups.round(3).to_string(index=False))

In [ ]:
def gap_row(name, block):
    low, high = block["confidence_interval"]
    return {"criterion": name, "gap": round(block["gap"], 3), "95% CI": f"[{low:.3f}, {high:.3f}]"}

eo = fair["equalized_odds"]
table = pd.DataFrame([
    gap_row("statistical parity (SR)", fair["statistical_parity"]),
    gap_row("equalized odds: TPR", eo["true_positive_rate_gap"]),
    gap_row("equalized odds: FPR", eo["false_positive_rate_gap"]),
    gap_row("predictive parity (PPV)", fair["predictive_parity"]),
])
print(table.to_string(index=False))
print("equalized-odds maximum gap:", round(eo["maximum_gap"], 3))

fig, ax = plt.subplots(figsize=(7, 3.8))
metrics = ["selection_rate", "true_positive_rate", "false_positive_rate", "positive_predictive_value"]
width = 0.8 / len(groups)
for i, row in groups.iterrows():
    ax.bar(np.arange(len(metrics)) + i * width, [row[m] for m in metrics], width, label=f"{row['group']} (n={row['sample_count']})")
ax.set_xticks(np.arange(len(metrics)) + width / 2, ["SR", "TPR", "FPR", "PPV"]); ax.set_ylim(0, 1)
ax.set_title("Per-group rates"); ax.legend(); plt.show()

### Verify the rates by hand (pandas)

In [ ]:
pred = pipe.predict(Xte)
frame = pd.DataFrame({"A": Ate, "y": yte, "pred": pred})
manual = frame.groupby("A").apply(lambda d: pd.Series({
    "n": len(d), "SR": (d.pred == 1).mean(),
    "TPR": ((d.pred == 1) & (d.y == 1)).sum() / (d.y == 1).sum(),
    "FPR": ((d.pred == 1) & (d.y == 0)).sum() / (d.y == 0).sum(),
    "PPV": ((d.pred == 1) & (d.y == 1)).sum() / (d.pred == 1).sum()}), include_groups=False)
print(manual.round(3))

## 3. Real data: breast cancer with an illustrative group

The group below (tumour size above/below the median) is **not** a protected attribute; it only demonstrates the API on a bundled dataset and the effect of small denominators (few actual negatives in one group make FPR undefined or very noisy). Two attributes can be audited in one call.

In [ ]:
bc = load_breast_cancer(as_frame=True)
Xb = bc.data[["mean radius", "mean texture", "mean smoothness", "mean compactness", "mean symmetry"]]
yb = bc.target
Xbtr, Xbte, ybtr, ybte = train_test_split(Xb, yb, test_size=0.4, random_state=0, stratify=yb)
size = np.where(Xbte["mean radius"] > Xbte["mean radius"].median(), "large", "small")
texture = np.where(Xbte["mean texture"] > Xbte["mean texture"].median(), "rough", "smooth")
bpipe = ReliablePipeline(RandomForestClassifier(100, random_state=0), plot_dir="reliability_plots",
                         bias_variance_folds=3, bias_variance_repeats=1, bootstrap_samples=200, random_seed=0).fit(Xbtr, ybtr)
bres = bpipe.evaluate(Xbte, ybte, sensitive_features={"size": size, "texture": texture}, positive_label=1).to_dict()["fairness"]
for attribute, block in bres.items():
    cls = block["classes"]["1"]
    print(f"--- attribute: {attribute}")
    print(pd.DataFrame(cls["per_group"])[["group", "sample_count", "actual_positive_count", "actual_negative_count",
          "selection_rate", "true_positive_rate", "false_positive_rate"]].round(3).to_string(index=False))
    print("parity gap", round(cls["statistical_parity"]["gap"], 3), "CI", [round(v, 3) for v in cls["statistical_parity"]["confidence_interval"]],
          "| equalized-odds max gap", round(cls["equalized_odds"]["maximum_gap"], 3))

## 4. Multiclass (wine), one-vs-rest per class

In [ ]:
wine = load_wine(as_frame=True)
Xw = wine.data[["alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium", "flavanoids"]]
Xwtr, Xwte, ywtr, ywte = train_test_split(Xw, wine.target, test_size=0.5, random_state=0, stratify=wine.target)
batch = np.where(Xwte["alcohol"] > Xwte["alcohol"].median(), "high_alcohol", "low_alcohol")
wpipe = ReliablePipeline(LogisticRegression(max_iter=2000), plot_dir="reliability_plots",
                         bias_variance_folds=3, bias_variance_repeats=1, bootstrap_samples=200, random_seed=0).fit(Xwtr, ywtr)
wres = wpipe.evaluate(Xwte, ywte, sensitive_features={"alcohol_group": batch}).to_dict()["fairness"]["alcohol_group"]
for label, cls in wres["classes"].items():
    print(f"class {label}: parity gap = {cls['statistical_parity']['gap']:.3f}, "
          f"equalized-odds max gap = {cls['equalized_odds']['maximum_gap']}")

## Interpretation and limitations

**How to read the output.** Read the per-group table first: the denominators (sample, actual-positive, actual-negative counts) tell you how much each rate can be trusted. The intervals around each gap show whether an observed disparity is distinguishable from sampling noise; the synthetic example shows that excluding the sensitive attribute from the model does not remove group differences when a proxy feature and different base rates exist, and that the criteria give gaps of different size (they measure different things).

**Limitations.** Gaps are observational and descriptive: they do not identify causes, justify a legal or ethical conclusion, or tell you which fairness criterion should apply. Intervals are within-group row bootstraps (independent rows assumed), no multiplicity correction is applied across criteria, attributes or classes, and small or empty cells give undefined or unstable rates. Intersectional groups are not formed automatically. Labels may themselves be biased.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.